# **Reservoir Capacity Forecasting — Exploratory Data Analysis**

## **Objective**
Understand the statistical properties of the Madrid reservoir system before 
modelling. Key questions:
- What does the historical capacity profile look like?
- Is there a clear seasonal pattern?
- Which reservoirs dominate the system?
- What are meaningful drought thresholds?
- Is the series stationary? What ARIMA order does the ACF/PACF suggest?

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from statsmodels.tsa.stattools import adfuller, acf, pacf

PROJECT_ROOT = Path.cwd().parent
DATA_PROC    = PROJECT_ROOT / "data" / "processed"

df_pivot = pd.read_csv(DATA_PROC / "reservoirs_pivot.csv", parse_dates=["ds"])
df_long  = pd.read_csv(DATA_PROC / "reservoirs_long.csv")

RESERVOIR_COLS = [c for c in df_pivot.columns if c not in ["ds", "year", "month", "total_hm3"]]

print(f"Pivot shape : {df_pivot.shape}")
print(f"Long shape  : {df_long.shape}")
print(f"Date range  : {df_pivot['ds'].min().date()} → {df_pivot['ds'].max().date()}")
print(f"Reservoirs  : {len(RESERVOIR_COLS)}")

Pivot shape : (279, 17)
Long shape  : (3627, 4)
Date range  : 1998-01-01 → 2021-03-01
Reservoirs  : 13


## **1. Historical Series**

Total system capacity 1998–2021. Mean 661 hm³, median 667 hm³.
The 2005–2006 drought is clearly visible — the system dropped to 329 hm³, 
its lowest point in the 23-year record.

In [2]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_pivot["ds"],
    y=df_pivot["total_hm3"],
    mode="lines",
    name="Total system",
    line=dict(color="#4a9eff", width=2),
    fill="tozeroy",
    fillcolor="rgba(74, 158, 255, 0.08)",
))

fig.update_layout(
    title="Madrid Reservoir System — Total Capacity 1998–2021",
    xaxis_title="Date",
    yaxis_title="Capacity (hm³)",
    template="plotly_dark",
    height=450,
    hovermode="x unified",
    width=1200,
)

fig.show()

print(f"Mean   : {df_pivot['total_hm3'].mean():,.1f} hm³")
print(f"Median : {df_pivot['total_hm3'].median():,.1f} hm³")
print(f"Min    : {df_pivot['total_hm3'].min():,.1f} hm³  ({df_pivot.loc[df_pivot['total_hm3'].idxmin(), 'ds'].date()})")
print(f"Max    : {df_pivot['total_hm3'].max():,.1f} hm³  ({df_pivot.loc[df_pivot['total_hm3'].idxmax(), 'ds'].date()})")

Mean   : 661.8 hm³
Median : 667.4 hm³
Min    : 329.2 hm³  (2005-10-01)
Max    : 907.9 hm³  (2011-05-01)


## **2. Seasonality**

Average capacity by month shows a consistent annual cycle driven by 
Mediterranean climate patterns:
- **Peak: April–May** — snowmelt from Sierra de Guadarrama + spring rainfall
- **Trough: September–October** — end of the dry summer season

This strong, stable seasonality justifies using seasonal models (SARIMAX, Prophet)
and cyclic encoding (month_sin, month_cos) for ML models.

In [3]:
# Average capacity by month across all years
monthly_avg = (
    df_pivot.groupby("month")["total_hm3"]
    .mean()
    .reindex(["enero", "febrero", "marzo", "abril", "mayo", "junio",
              "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"])
)

fig = go.Figure(go.Bar(
    x=monthly_avg.index,
    y=monthly_avg.values,
    marker_color="#4a9eff",
    opacity=0.85,
))

fig.update_layout(
    title="Average Reservoir Capacity by Month (1998–2021)",
    xaxis_title="Month",
    yaxis_title="Capacity (hm³)",
    template="plotly_dark",
    height=400,
    width=1200,
)

fig.show()

print("Peak months   :", monthly_avg.nlargest(3).index.tolist())
print("Trough months :", monthly_avg.nsmallest(3).index.tolist())

Peak months   : ['mayo', 'abril', 'junio']
Trough months : ['octubre', 'noviembre', 'septiembre']


## **3. Annual Distribution**

Violin plots by year reveal:
- **Driest years: 2002, 2005, 2006** — means below 500 hm³
- **Wettest years: 2011, 2013, 2021** — means above 750 hm³ (2021 has only 3 months)
- High interannual variability — the system can swing 400 hm³ between years

In [4]:
annual_stats = df_pivot.groupby("year")["total_hm3"].agg(["mean", "min", "max"]).reset_index()

fig = go.Figure()

for year in sorted(df_pivot["year"].unique()):
    values = df_pivot[df_pivot["year"] == year]["total_hm3"].values
    fig.add_trace(go.Violin(
        x=[year] * len(values),
        y=values,
        name=str(year),
        box_visible=True,
        meanline_visible=True,
        showlegend=False,
        line_color="#4a9eff",
        fillcolor="rgba(74, 158, 255, 0.15)",
    ))

fig.update_layout(
    title="Annual Distribution of Total Reservoir Capacity",
    xaxis_title="Year",
    yaxis_title="Capacity (hm³)",
    template="plotly_dark",
    height=500,
    xaxis=dict(type="category"),
    width=1500,
)

fig.show()

# Driest and wettest years
print("Driest years  :", annual_stats.nsmallest(3, "mean")[["year", "mean"]].to_string(index=False))
print("Wettest years :", annual_stats.nlargest(3, "mean")[["year", "mean"]].to_string(index=False))

Driest years  :  year       mean
 2002 453.802333
 2005 465.380083
 2006 484.171167
Wettest years :  year    mean
 2021 793.788
 2011 780.407
 2013 757.670


## **4. Reservoir Contribution**

The system is heavily concentrated:
- **RioLozoya_ElAtazar: 47.3%** of total capacity
- Top 2 reservoirs (ElAtazar + Valmayor): 60.7%

This means the total system forecast is essentially a forecast of ElAtazar 
with corrections from the other 12 reservoirs.

In [5]:
# Mean capacity per reservoir
reservoir_means = df_pivot[RESERVOIR_COLS].mean().sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=reservoir_means.index,
    y=reservoir_means.values,
    marker_color="#4a9eff",
    opacity=0.85,
    text=[f"{v:.0f}" for v in reservoir_means.values],
    textposition="outside",
    textfont=dict(size=10),
))

fig.update_layout(
    title="Mean Capacity by Reservoir (1998–2021)",
    xaxis_title="Reservoir",
    yaxis_title="Mean Capacity (hm³)",
    template="plotly_dark",
    height=450,
    xaxis=dict(tickangle=-45),
    margin=dict(b=120),
    width=1200,
)

fig.show()

# Share of total
total_mean = reservoir_means.sum()
print("Reservoir contribution to total system:")
for name, val in reservoir_means.items():
    print(f"  {name:<45} {val:>6.1f} hm³  ({val/total_mean*100:.1f}%)")

Reservoir contribution to total system:
  RioLozoya_ElAtazar                             313.1 hm³  (47.3%)
  RioGuadarrama-Aulencia_Valmayor                 88.4 hm³  (13.4%)
  RioManzanares_Santillana                        64.2 hm³  (9.7%)
  RioLozoya_PuentesViejas                         37.3 hm³  (5.6%)
  RioLozoya_Riosequillo                           36.5 hm³  (5.5%)
  RioLozoya_LaPinilla                             27.7 hm³  (4.2%)
  RioJarama_ElVado                                26.5 hm³  (4.0%)
  RioGuadalix_Pedrezuela                          23.9 hm³  (3.6%)
  RioLozoya_ElVillar                              17.8 hm³  (2.7%)
  RioCofio_LaAcena                                14.9 hm³  (2.3%)
  RioManzanares_Navacerrada                        6.4 hm³  (1.0%)
  RioGuadarrama-Aulencia_LaJorosa                  4.7 hm³  (0.7%)
  RioGuadarrama-Aulencia_Navalmedio                0.4 hm³  (0.1%)


## **5. Drought Thresholds**

Thresholds derived empirically from the historical distribution:
- **Severe drought: < 488 hm³ (p10)** — 28 months in the historical record (10%)
- **Moderate drought: < 574 hm³ (p25)** — 42 additional months (15%)
- **Good conditions: > 772 hm³ (p75)**

These thresholds are used in the Streamlit app to classify forecast months.

In [6]:
DROUGHT_SEVERE   = float(df_pivot["total_hm3"].quantile(0.10))
DROUGHT_MODERATE = float(df_pivot["total_hm3"].quantile(0.25))
NORMAL           = float(df_pivot["total_hm3"].quantile(0.75))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_pivot["ds"],
    y=df_pivot["total_hm3"],
    mode="lines",
    name="Total system",
    line=dict(color="#4a9eff", width=1.5),
))

fig.add_hline(y=DROUGHT_SEVERE,   line=dict(color="#ff4444", dash="dot", width=1.5),
            annotation_text=f"Severe drought  {DROUGHT_SEVERE:.0f} hm³ (p10)",
            annotation_font=dict(color="#ff4444"))
fig.add_hline(y=DROUGHT_MODERATE, line=dict(color="#ffaa44", dash="dot", width=1.5),
            annotation_text=f"Moderate drought  {DROUGHT_MODERATE:.0f} hm³ (p25)",
            annotation_font=dict(color="#ffaa44"))
fig.add_hline(y=NORMAL,           line=dict(color="#44ff88", dash="dot", width=1.5),
            annotation_text=f"Good  {NORMAL:.0f} hm³ (p75)",
            annotation_font=dict(color="#44ff88"))

fig.update_layout(
    title="Reservoir Capacity with Drought Thresholds",
    xaxis_title="Date",
    yaxis_title="Capacity (hm³)",
    template="plotly_dark",
    height=450,
    hovermode="x unified",
    width=1200,
)

fig.show()

# Count months in each zone
n_severe   = (df_pivot["total_hm3"] < DROUGHT_SEVERE).sum()
n_moderate = ((df_pivot["total_hm3"] >= DROUGHT_SEVERE) & (df_pivot["total_hm3"] < DROUGHT_MODERATE)).sum()
n_normal   = (df_pivot["total_hm3"] >= DROUGHT_MODERATE).sum()

print(f"Severe drought   (<{DROUGHT_SEVERE:.0f} hm³) : {n_severe:>3} months ({n_severe/len(df_pivot)*100:.1f}%)")
print(f"Moderate drought (<{DROUGHT_MODERATE:.0f} hm³) : {n_moderate:>3} months ({n_moderate/len(df_pivot)*100:.1f}%)")
print(f"Normal/good      (>{DROUGHT_MODERATE:.0f} hm³) : {n_normal:>3} months ({n_normal/len(df_pivot)*100:.1f}%)")

Severe drought   (<488 hm³) :  28 months (10.0%)
Moderate drought (<574 hm³) :  42 months (15.1%)
Normal/good      (>574 hm³) : 209 months (74.9%)


## **6. Stationarity — ADF Test**

Augmented Dickey-Fuller test: ADF = -4.91, p = 0.000034.

The series is **stationary** — reject H0 at 1% significance. No unit root.
This means SARIMAX can be applied with `d=0` (no non-seasonal differencing needed).

The series appears stationary because capacity oscillates around a stable mean 
driven by the annual hydrological cycle, without a long-term trend.

In [7]:
result = adfuller(df_pivot["total_hm3"].dropna(), autolag="AIC")

print("Augmented Dickey-Fuller Test")
print(f"  ADF Statistic : {result[0]:.4f}")
print(f"  p-value       : {result[1]:.4f}")
print(f"  Lags used     : {result[2]}")
print(f"  Observations  : {result[3]}")
print()
print("Critical values:")
for key, val in result[4].items():
    print(f"  {key}: {val:.4f}")
print()
if result[1] < 0.05:
    print("✓ Series is STATIONARY (reject H0 at 5%)")
    print("  SARIMAX can be applied directly — no differencing needed")
else:
    print("✗ Series is NON-STATIONARY (fail to reject H0)")
    print("  SARIMAX will need d=1 differencing")

Augmented Dickey-Fuller Test
  ADF Statistic : -4.9063
  p-value       : 0.0000
  Lags used     : 12
  Observations  : 266

Critical values:
  1%: -3.4552
  5%: -2.8725
  10%: -2.5726

✓ Series is STATIONARY (reject H0 at 5%)
  SARIMAX can be applied directly — no differencing needed


## **7. ACF / PACF — Model Order Selection**

**ACF** — slow decay in lags 1–5, significant spikes at lags 15–20 and 33–36.
Pattern consistent with an AR process with seasonal component.

**PACF** — dominant lag 1 (0.92), significant negative lag 2 (-0.58), 
then near-zero until lag 12.

**Implied SARIMAX order: (2,0,1)(1,1,1,12)**
- `p=2` — PACF cuts off after lag 2
- `d=0` — series is stationary
- `q=1` — ACF decays gradually
- `P=1, D=1, Q=1, s=12` — annual seasonal cycle

In [8]:
n_lags = 36  # 3 years

acf_values  = acf(df_pivot["total_hm3"].dropna(),  nlags=n_lags)
pacf_values = pacf(df_pivot["total_hm3"].dropna(), nlags=n_lags)

conf_interval = 1.96 / np.sqrt(len(df_pivot))

fig = make_subplots(rows=2, cols=1, subplot_titles=("ACF — Autocorrelation", "PACF — Partial Autocorrelation"))

# ACF
fig.add_trace(go.Bar(
    x=list(range(n_lags + 1)),
    y=acf_values,
    marker_color="#4a9eff",
    name="ACF",
), row=1, col=1)

# PACF
fig.add_trace(go.Bar(
    x=list(range(n_lags + 1)),
    y=pacf_values,
    marker_color="#4a9eff",
    name="PACF",
    showlegend=False,
), row=2, col=1)

# Confidence intervals
for row in [1, 2]:
    fig.add_hline(y= conf_interval, line=dict(color="#ff4444", dash="dash", width=1), row=row, col=1)
    fig.add_hline(y=-conf_interval, line=dict(color="#ff4444", dash="dash", width=1), row=row, col=1)
    fig.add_hline(y=0,              line=dict(color="#666",    dash="solid", width=1), row=row, col=1)

fig.update_layout(
    template="plotly_dark",
    height=600,
    title="Autocorrelation Analysis — Total Reservoir Capacity",
    showlegend=False,
    width=1200,
)
fig.update_xaxes(title_text="Lag (months)")
fig.update_yaxes(title_text="Correlation")

fig.show()

# Key lags
sig_acf  = [i for i, v in enumerate(acf_values[1:], 1)  if abs(v) > conf_interval]
sig_pacf = [i for i, v in enumerate(pacf_values[1:], 1) if abs(v) > conf_interval]
print(f"Significant ACF lags  : {sig_acf}")
print(f"Significant PACF lags : {sig_pacf}")
print()
print(f"ACF  at lag 12 : {acf_values[12]:.4f}  ← seasonal signal")
print(f"ACF  at lag 24 : {acf_values[24]:.4f}  ← 2-year cycle")
print(f"PACF at lag  1 : {pacf_values[1]:.4f}  ← AR(1) component")
print(f"PACF at lag 12 : {pacf_values[12]:.4f}  ← SAR(1) component")

Significant ACF lags  : [1, 2, 3, 4, 5, 10, 11, 12, 15, 16, 17, 18, 19, 20, 23, 24, 25, 28, 29, 30, 31, 33, 34, 35, 36]
Significant PACF lags : [1, 2, 5, 6, 11, 12, 14, 19, 20, 24, 29, 34]

ACF  at lag 12 : 0.1742  ← seasonal signal
ACF  at lag 24 : 0.2097  ← 2-year cycle
PACF at lag  1 : 0.9226  ← AR(1) component
PACF at lag 12 : -0.1919  ← SAR(1) component


## **8. Validation & Export**

In [9]:
# Export EDA constants for use in notebook 03
eda_summary = {
    "drought_thresholds": {
        "severe_hm3":   round(DROUGHT_SEVERE,   1),
        "moderate_hm3": round(DROUGHT_MODERATE, 1),
        "good_hm3":     round(NORMAL,           1),
    },
    "system_stats": {
        "mean_hm3":   round(float(df_pivot["total_hm3"].mean()),   1),
        "median_hm3": round(float(df_pivot["total_hm3"].median()), 1),
        "min_hm3":    round(float(df_pivot["total_hm3"].min()),    1),
        "max_hm3":    round(float(df_pivot["total_hm3"].max()),    1),
    },
    "seasonality": {
        "peak_months":   ["mayo", "abril", "junio"],
        "trough_months": ["octubre", "noviembre", "septiembre"],
    },
    "stationarity": {
        "adf_statistic": round(result[0], 4),
        "p_value":       round(result[1], 6),
        "is_stationary": bool(result[1] < 0.05),
    },
    "sarimax_order": {
        "order":         [2, 0, 1],
        "seasonal_order": [1, 1, 1, 12],
    },
}

with open(DATA_PROC / "eda_summary.json", "w") as f:
    json.dump(eda_summary, f, indent=2)

print("Exported eda_summary.json")
print(json.dumps(eda_summary, indent=2))

Exported eda_summary.json
{
  "drought_thresholds": {
    "severe_hm3": 488.5,
    "moderate_hm3": 573.6,
    "good_hm3": 772.5
  },
  "system_stats": {
    "mean_hm3": 661.8,
    "median_hm3": 667.4,
    "min_hm3": 329.2,
    "max_hm3": 907.9
  },
  "seasonality": {
    "peak_months": [
      "mayo",
      "abril",
      "junio"
    ],
    "trough_months": [
      "octubre",
      "noviembre",
      "septiembre"
    ]
  },
  "stationarity": {
    "adf_statistic": -4.9063,
    "p_value": 3.4e-05,
    "is_stationary": true
  },
  "sarimax_order": {
    "order": [
      2,
      0,
      1
    ],
    "seasonal_order": [
      1,
      1,
      1,
      12
    ]
  }
}
